In [0]:
create table if not exists cloud_pricing.silver.vm_prices (
price_key string, run_id string, run_date date, observed_at timestamp,
region string, location string,
meter_id string, meter_name string, sku_id string, sku_name string, arm_sku_name string,
product_id string, product_name string,
price_type string, term string, os string, capacity_type string,
unit_of_measure string, currency_code string, tier_min_units decimal(20,4),
retail_price decimal(20,10), effective_start_date date,
is_primary_region boolean, ingested_at timestamp
)

In [0]:
-- only runs not in silver yet
create or replace temp view parsed as
select b.run_id, b.run_date, b.ingested_at,
from_json(b.item, 'currencyCode STRING, tierMinimumUnits DECIMAL(20,4), reservationTerm STRING, retailPrice DECIMAL(20,10), armRegionName STRING, location STRING, effectiveStartDate STRING, meterId STRING, meterName STRING, productId STRING, skuId STRING, productName STRING, skuName STRING, armSkuName STRING, unitOfMeasure STRING, type STRING, isPrimaryMeterRegion BOOLEAN, savingsPlan ARRAY<STRUCT<retailPrice: DECIMAL(20,10), term: STRING>>') p
from cloud_pricing.bronze.retail_prices_raw b
where b.run_id not in (select distinct run_id from cloud_pricing.silver.vm_prices)

In [0]:
create or replace temp view priced as
select run_id, run_date, ingested_at, p, p.type price_type, p.reservationTerm term, p.retailPrice retail_price
from parsed
union all
-- savings plan prices sit inside the payg row, pulling them out as their own rows
select run_id, run_date, ingested_at, p, 'SavingsPlan', sp.term, sp.retailPrice
from (select *, explode(p.savingsPlan) sp from parsed)

In [0]:
insert into cloud_pricing.silver.vm_prices
select
sha2(concat_ws('|', p.meterId, p.skuId, price_type, coalesce(term, ''), cast(p.tierMinimumUnits as string), p.currencyCode, p.armRegionName), 256) price_key,
run_id, run_date, to_timestamp(run_id, "yyyyMMdd'T'HHmmss'Z'") observed_at,
p.armRegionName region, p.location location,
p.meterId meter_id, p.meterName meter_name, p.skuId sku_id, p.skuName sku_name, p.armSkuName arm_sku_name,
p.productId product_id, p.productName product_name,
price_type, term,
-- api has no os field, windows products end with "Windows" in the name
case when p.productName like '%Windows' then 'windows' else 'linux' end os,
case when p.meterName like '%Spot%' then 'spot' when p.meterName like '%Low Priority%' then 'low_priority' else 'regular' end capacity_type,
p.unitOfMeasure unit_of_measure, p.currencyCode currency_code, p.tierMinimumUnits tier_min_units, retail_price,
to_date(p.effectiveStartDate) effective_start_date, p.isPrimaryMeterRegion is_primary_region, ingested_at
from priced

In [0]:
-- fails the notebook if any run has duplicate keys
select run_id, assert_true(count(*) = count(distinct price_key), concat('duplicate price keys in ', run_id))
from cloud_pricing.silver.vm_prices
group by run_id

In [0]:
-- every bronze item should be in silver, savings plan rows are extra
select b.run_id, b.bronze_items, s.silver_items, s.savings_plan_rows, s.null_prices, s.non_usd, s.other_units
from (select run_id, count(*) bronze_items from cloud_pricing.bronze.retail_prices_raw group by run_id) b
left join (
select run_id,
sum(case when price_type <> 'SavingsPlan' then 1 else 0 end) silver_items,
sum(case when price_type = 'SavingsPlan' then 1 else 0 end) savings_plan_rows,
sum(case when retail_price is null then 1 else 0 end) null_prices,
sum(case when currency_code <> 'USD' then 1 else 0 end) non_usd,
sum(case when unit_of_measure <> '1 Hour' then 1 else 0 end) other_units
from cloud_pricing.silver.vm_prices group by run_id
) s on b.run_id = s.run_id
order by b.run_id